In [3]:
import sys
sys.path.append('../src/')
from xpcs import *
from plotting import *
from autocorrelations import *
import cv2
from scipy.special import erfinv
import pyopencl as cl
from pathlib import Path
import matplotlib.cm as cm
import json
import matplotlib.ticker as ticker
#from cmcrameri import cm
import matplotlib as mpl

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import ListedColormap
from matplotlib.colors import LinearSegmentedColormap


%matplotlib notebook

In [9]:


# OPTIONAL (recommended): perceptual interpolation
# pip install colorspacious
try:
    from colorspacious import cspace_convert
    USE_CS = True
except Exception:
    USE_CS = False

def uniformized_slice_from_nipy(start=0.10, stop=0.80, N=256, name="nipy_slice_uniform"):
    """
    Take a subrange of nipy_spectral and re-interpolate it uniformly in a perceptual space.
    start/stop are in [0,1] along the original colormap.
    """
    base = plt.colormaps["nipy_spectral"]
    # sample the chosen slice
    t = np.linspace(start, stop, 512)
    rgb = base(t)[:, :3]  # drop alpha

    if USE_CS:
        # Convert to CAM02-UCS (nearly uniform)
        jab = cspace_convert(rgb, "sRGB1", "CAM02-UCS")
        # Arc-length parameterization → equal steps in perceived distance
        seg = np.cumsum(np.r_[0, np.linalg.norm(np.diff(jab, axis=0), axis=1)])
        seg /= seg[-1]
        # Resample at equal perceptual spacing
        new_t = np.linspace(0, 1, N)
        jab_u = np.column_stack([np.interp(new_t, seg, jab[:,i]) for i in range(3)])
        rgb_u = np.clip(cspace_convert(jab_u, "CAM02-UCS", "sRGB1"), 0, 1)
    else:
        # Fallback: simple (non-perceptual) resample in RGB
        # (still uses your slice, just not perfectly uniform)
        new_t = np.linspace(0, 1, N)
        idx = (start + new_t*(stop-start))
        rgb_u = base(idx)[:, :3]

    return ListedColormap(rgb_u, name=name)

# Example: green→cyan→blue→magenta slice (tweak start/stop to taste)
nipy_uc = uniformized_slice_from_nipy(start=0.02, stop=0.85, N=256, name="nipy_uc")

import numpy as np
from matplotlib.colors import LinearSegmentedColormap

# anchors pulled from your g2 categorical palette
#anchors = ['#185f69','#24ab97','#a3c3d4','#e3bccd','#ffcfc4','#e8a0ac','#ba8fa8']   # dark teal -> pale -> warm pink
anchors = ['#4a96a8','#a3c3d4','#e8a0ac','#ba8fa8','#5a3a52']
nps_seq = LinearSegmentedColormap.from_list('nps_seq', anchors, N=256)

# pip install viscm  — or use colorspacious to check uniformity
# Quick check without extra tools: sample the cmap and verify L* rises
from colorspacious import cspace_convert
samples = nps_seq(np.linspace(0, 1, 64))[:, :3]
L = cspace_convert(samples, "sRGB1", "CAM02-UCS")[:, 0]


from matplotlib.colors import LinearSegmentedColormap




nps_custom = LinearSegmentedColormap.from_list(
  'nps_custom', [
    (0.00, '#000000'),
    (0.12, '#342c7a'),
    (0.25, '#a43479'),
    (0.55, '#d74444'),
    (0.78, '#f3de80'),
    (1.00, '#ffffff'),
], N=256
)



In [5]:
### Figure Settings ###

plt.rcParams['font.sans-serif'] = ['Helvetica']

legend_ax_font = 12
ax_label_font = 16
ax_title_font = 18

minor_tick_length = 6
major_tick_length = 9
minor_tick_width = 0.75
major_tick_width = 1.0

marker_size = 8
fig_size = (8,5)

nps_custom = LinearSegmentedColormap.from_list(
  'nps_custom', [
    (0.00, '#000000'),
    (0.07, '#342c7a'), #0.12
    (0.20, '#a43479'), #25
    (0.55, '#d74444'),
    (0.78, '#f3de80'),
    (1.00, '#ffffff'),
], N=256
)

In [ ]:
# Define the path to the configuration file
config_path = Path("../configs/config_NPS_example.json")

# Open and load the JSON configuration file
with open(config_path, "r") as file:
    config = json.load(file)
    #json_str = json.loads(jsonString, strict=False)

### Output Directory ###
figure_path = config_path.parent.parent / config.get("Figures")
figure_path.mkdir(parents=True, exist_ok=True)

    
temp298 = Path(config.get("219_temperature"))
temp305 = Path(config.get("334_temperature"))
temp325 = Path(config.get("552_temperature"))
temp345 = Path(config.get("579_temperature"))
temp365 = Path(config.get("709_temperature"))
temp385 = Path(config.get("779_temperature"))
temp405 = Path(config.get("796_temperature"))
temp350 = Path(config.get("868_temperature"))

scans = np.array([219,334,552,579,709,779,796,868])

#T = np.array([298,305,325,365,385,405,350])
T = np.array([298,305,325,345,365,385,405,350])

best_ind = np.array([0,0,0,0,0,0,0,0])


In [10]:
avgG2 = []
avgG2_err = []
avgtaus = []
avgbetas = []
avgAs = []
avgaas = []
all_delays = []
TwoTimes = []
TT_exts = []

first_ring = 0
last_ring = 3

fig1,ax = plt.subplots(figsize=(8.5,5))

# --- sizes ---
ax.tick_params(which='major', length=major_tick_length, width=major_tick_width, labelsize=ax_label_font)
ax.tick_params(which='minor', length=minor_tick_length, width=minor_tick_width,labelsize=ax_label_font)

# --- directions per side ---
# x: top IN, bottom OUT (two calls so we don't overwrite the other side)
ax.tick_params(axis='x', which='both',direction='in', top=True)
ax.tick_params(axis='x', which='both',direction='out', bottom=True)

# y: right IN, left OUT
ax.tick_params(axis='y', which='both',direction='out', right=True)
ax.tick_params(axis='y', which='both',direction='out', left=True)

#TWO-TIME Plotting
fig2,axs = plt.subplots(2,4,figsize=(25,25))

#Individual Plots

fig3,ax3 = plt.subplots(figsize=(16,10))
fig4,ax4 = plt.subplots(figsize=(5,5))

##cmap = cm.Set2 # or any other colormap
#color_list = [cmap(i) for i in np.linspace(0, 1, avgG2.shape[0])] # num_colors is the desired number of colors
#color_list = np.array(['red','blue','gray','orange','magenta','black','green','purple'])

#Wada Colors
#np.array(['#185f69','#24ab97','#a3c3d4','#e3bccd','#ffcfc4','#e8a0ac','#ba8fa8','#a1a7ab'])
#color_list = np.array(['#185f69','#4a96a8','#a3c3d4','#c3bfd0','#e3bccd','#e8a0ac','#ba8fa8','#a1a7ab'])
color_list = np.array(['#074a54','#1c94b0','#6abce7','#9c86e3','#ee8bb6','#f27086','#da499d','#7e8285'])
marker_list = np.array(['v','s','o','*','p','D','>','H'])

#cmap = plt.cm.get_cmap('GnBu', len(T))
cmap = nps_seq

norm = mpl.colors.Normalize(vmin=T.min(), vmax=T.max())
#color_list = mpl.cm.ScalarMappable(norm=norm, cmap=cmap).to_rgba

#color_list = cmap(np.linspace(0, 1, len(T)+1))  # spread evenly across the colormap

POPT = np.zeros((T.shape[0],3))
PCOV = np.zeros((T.shape[0],3))

j = 0
k = 0

base_path = Path(config.get("Base")) 
results_path = Path(config.get("Results"))

for i,(TT,ss) in enumerate(zip(T,scans)):
        
    file = base_path / Path(config.get(f"{ss}")) / results_path / Path(config.get(f"{ss}_results_3"))
    #Load file
    with h5py.File(file, 'r') as DATA:
        g2s = DATA['XPCS/G2_Results'][:]
        g2_errs = DATA['XPCS/G2_error'][:]
        delays = DATA['XPCS/delays'][:]
        dQ = DATA['XPCS/dQ'][:]
        As = DATA['XPCS/As'][:][0]
        aas = DATA['XPCS/aas'][:][0]

       # Cs = DATA['XPCS/Cs'][:]
        taus = DATA['XPCS/taus'][:]
        betas = DATA['XPCS/betas'][:]
        taus_err = DATA['XPCS/taus_err'][:]
        betas_err = DATA['XPCS/betas_err'][:]
        
        #TwoTime
        TwoTime = DATA['XPCS/TwoTime'][:]
        #print(TwoTime.shape)
        TwoTime_extent = DATA['XPCS/TwoTime_extent'][:]
        
        #print(As)
        truncs = -5
        # Fit

        if ss==219:
            truncs = -32
            mask = delays[0,:] <= delays[0,truncs]
            print(fr"$\tau$ max scan {ss}: {delays[0,truncs]}")
        if ss==334:
            truncs = -20
            mask = delays[0,:] <= delays[0,truncs]
            print(fr"$\tau$ max scan {ss}: {delays[0,truncs]}")
        if ss==552:
            truncs = -20
            mask = delays[0,:] <= delays[0,truncs]
            print(fr"$\tau$ max scan {ss}: {delays[0,truncs]}")
        if ss==579:
            truncs = -18
            mask = delays[0,:] <= delays[0,truncs]
            print(fr"$\tau$ max scan {ss}: {delays[0,truncs]}")
        if ss==709:
            truncs = -23
            mask = delays[0,:] <= delays[0,truncs]
            print(fr"$\tau$ max scan {ss}: {delays[0,truncs]}")
        if ss==779:
            truncs = -10
            mask = delays[0,:] <= delays[0,truncs]
            print(fr"$\tau$ max scan {ss}: {delays[0,truncs]}")
        if ss==796:
            truncs = -11
            mask = delays[0,:] <= delays[0,truncs]
            print(fr"$\tau$ max scan {ss}: {delays[0,truncs]}")
        if ss==868:
            truncs = -16  #-16
            mask = delays[0,:] <= delays[0,truncs]
            print(fr"$\tau$ max scan {ss}: {delays[0,truncs]}")


        
        #popt, pcov = curve_fit(g2_two_tau, delays[0,:truncs],g2s[best_ind[i],:truncs], p0=[As,aas,taus[0],taus[1],betas[0],betas[1]],bounds=(lower_bounds, upper_bounds))

        #popt, pcov = curve_fit(g2_exp, delays[0,:truncs],g2s[best_ind[i],:truncs], p0=[As,taus[0],betas[0]])
        

        if ss==868:
            A = np.max(g2s[best_ind[i],:])-1
            popt, pcov = curve_fit(g2_exp, delays[0,mask],g2s[best_ind[i],mask], p0=[A,15000,1])
        
        if ss==779:
            A = np.max(g2s[best_ind[i],:])-1
            popt, pcov = curve_fit(g2_exp, delays[0,mask],g2s[best_ind[i],mask], p0=[A,1000,1.2])
        if ss==552:
            popt, pcov = curve_fit(g2_exp, delays[0,mask],g2s[best_ind[i],mask], p0=[As,50,2])
        if ss!=868 and ss!=552 and ss!=779:
            popt, pcov = curve_fit(g2_exp, delays[0,mask],g2s[best_ind[i],mask], p0=[As,taus[0],betas[0]])

        
    

        POPT[i,:]+=popt
        PCOV[i,:]+=np.sqrt(np.diag(pcov))
        

        ax.set_ylabel(r'$|F(\boldsymbol{Q},\tau)|^2$', fontsize=ax_label_font)
        ax.set_xlabel(r'$\tau$ (s)',fontsize=ax_label_font) 

        isf = (g2s[best_ind[i],:] - 1) / popt[0]
        isf_err = g2_errs[best_ind[i],:] / popt[0]
        isf_fit = ((g2_exp(delays[0,:],*popt)-1)/popt[0])[:]

            
        if ss==868:
            ax.semilogx(delays[0,:],isf,marker_list[i],markersize=5,markerfacecolor='none', markeredgecolor=color_list[i],label=f'{T[i]}K - Cooling',zorder=8-i)
        if ss!=65 and ss!=868:
            ax.semilogx(delays[0,:],isf,marker_list[i],markersize=5,color=color_list[i],label=f'{T[i]}K',zorder=8-i)

        ax.fill_between(delays[0,:], isf - isf_err, isf + isf_err,color=color_list[i], alpha=0.3, zorder=8-i)
        ax.semilogx(delays[0,:],isf_fit,'-',linewidth=2,color=color_list[i],zorder=8-i)

        #ax.errorbar(delays[0,:],((g2s[best_ind[i],:]-1)/popt[0])[:], yerr=g2_errs[best_ind[i],:],fmt='none',capsize=2,color=color_list[i])
        #ax.semilogx(delays[0,:],((g2_two_tau(delays[0,:],*popt)-1)/popt[0])[:],'-',color=color_list[i])
        #ax.semilogx(delays[0,:],((g2_exp(delays[0,:],*popt)-1)/popt[0])[:],'-',color=color_list[i],zorder=9-i)
        #ax.minorticks_on()
        #ax.grid(which='major', color='gray', linestyle=':', linewidth=0.8)
        #ax.grid(which='minor', color='gray', linestyle=':', linewidth=0.5)

        ax.set_xlim(1.75, 3050)
        #ax.legend(loc='lower left',fontsize=18)
        ax.set_ylim(bottom=-0.05)  # sets the lower limit to 0
        ax.set_ylim(top=1.1)  # sets the upper limit to 1.1

        leg = ax.legend(frameon=False,          # remove box
                handletextpad=0.3,      # space between marker and text
                handlelength=1.0,       # length of legend handle
                labelspacing=0.2,       # vertical spacing between entries
                borderpad=0.2,          # padding inside the legend area
                fontsize=legend_ax_font,
                loc='lower left')       # or wherever you want
        

        
        if ss==219:
                        # --- sizes ---
            ax3.tick_params(which='major', length=14, width=1.9, labelsize=22)
            ax3.tick_params(which='minor', length=11, width=1.5,labelsize=22)

            # --- directions per side ---
            # x: top IN, bottom OUT (two calls so we don't overwrite the other side)
            ax3.tick_params(axis='x', which='both',direction='in', top=True)
            ax3.tick_params(axis='x', which='both',direction='out', bottom=True)

            # y: right IN, left OUT
            ax3.tick_params(axis='y', which='both',direction='out', right=True)
            ax3.tick_params(axis='y', which='both',direction='out', left=True)



            ax3.set_ylabel(r'$|F(\boldsymbol{Q},\tau)|^2$', fontsize=32)
            ax3.set_xlabel(r'$\tau$ (s)',fontsize=32) 

            ax3.semilogx(delays[0,:],((g2s[best_ind[i],:]-1)/popt[0])[:],marker_list[i],markersize=8,color=color_list[i],label=f'{T[i]}K',zorder=8-i)

            ax3.semilogx(delays[0,:],((g2_exp(delays[0,:],*popt)-1)/popt[0])[:],'-',color=color_list[i],zorder=8-i)
            ax3.fill_between(delays[0,:], ((g2s[best_ind[i],:]-1)/popt[0])[:] - g2_errs[best_ind[i],:], ((g2s[best_ind[i],:]-1)/popt[0])[:] + g2_errs[best_ind[i],:], color=color_list[i], alpha=0.3,zorder=8-i)

            ax3.set_xlim(1.75, 3050)
            #ax.legend(loc='lower left',fontsize=18)
            ax3.set_ylim(bottom=-0.05)  # sets the lower limit to 0
            ax3.set_ylim(top=1.1)  # sets the upper limit to 1.1
        
            axt3, axr3 = asymmetric_ticks(ax3)
        
 
        #vmin = -0.22776,vmax = 1.2771
        if ss != 65:
                if i==4:
                    k=0
                if i>3:
                    j=1

                TwoTime_norm = TwoTime #(TwoTime-np.min(TwoTime))/(np.max(TwoTime)-np.min(TwoTime))
                vmin = 0#-0.2130334973335266
                vmax = 1#1.225314736366272
                im = axs[j,k].imshow(TwoTime_norm,origin="lower",extent=np.array(TwoTime_extent),vmin = vmin,vmax = vmax,cmap=nps_custom)
                
                if k==0:
                    axs[j,k].set_ylabel('$t_2$ (s)',fontsize=ax_label_font+2)
                if j==1:
                    axs[j,k].set_xlabel('$t_1$ (s)',fontsize=ax_label_font+2)
        
                #Tlabel = f'{T[i]} K'
                if TT != 350:
                    Tlabel = f'{TT}K'
                    color_label = color_list[i] #'#cc5e5e' #'red'
                if TT == 350:
                    Tlabel = f'{TT}K - Cooling'
                    color_label = color_list[i] #'#474273' #'cyan'
                    
                axs[j,k].text(0.024, 0.978, Tlabel, color=color_label, fontsize=28,fontweight='bold',
                    ha='left', va='top', transform=axs[j,k].transAxes,
                    bbox=dict(facecolor='white', alpha=1, edgecolor='none'))

                
                #if TT==350:
                #  axs[j,k].set_title(f'{T[i]} K, Cooling',fontsize=24)
                #else:
                #   axs[j,k].set_title(f'{T[i]} K' ,fontsize=24)


                #axs[j,k].xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x/1e3:.1f}"))
                #axs[j,k].yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f"{y/1e3:.1f}"))

                axs[j,k].xaxis.set_major_formatter(ticker.ScalarFormatter(useMathText=True))
                axs[j,k].xaxis.get_offset_text().set_fontsize(ax_label_font)
                axs[j,k].xaxis.get_offset_text().set_position((1.02, 0))  # Move offset label slightly

                axs[j,k].yaxis.set_major_formatter(ticker.ScalarFormatter(useMathText=True))
                axs[j,k].yaxis.get_offset_text().set_fontsize(ax_label_font)
                axs[j,k].yaxis.get_offset_text().set_position((0, 1.02))  # Move offset label slightly

                axs[j,k].ticklabel_format(axis='both', style='sci', scilimits=(0, 0))  # Force scientific notation

                ##axs[j,k].tick_params(axis='both', which='major', labelsize=20)
                ##axs[j,k].tick_params(axis='both', which='minor', labelsize=20)

                axs[j,k].tick_params(
                    which="major", length=major_tick_length, width=major_tick_width, labelsize=ax_label_font
                )
                axs[j,k].tick_params(
                    which="minor", length=minor_tick_length, width=minor_tick_width, labelsize=ax_label_font
                )
 
                if ss==219:
                    ax4.imshow(TwoTime_norm,origin="lower",extent=np.array(TwoTime_extent),vmin = 0,vmax = 1,cmap=nps_custom)

                    ax4.xaxis.set_major_formatter(ticker.ScalarFormatter(useMathText=True))
                    ax4.xaxis.get_offset_text().set_fontsize(20)  # Adjust font size if needed
                    ax4.xaxis.get_offset_text().set_position((1.02, 0))  # Move offset label slightly

                    ax4.yaxis.set_major_formatter(ticker.ScalarFormatter(useMathText=True))
                    ax4.yaxis.get_offset_text().set_fontsize(20)
                    ax4.yaxis.get_offset_text().set_position((0, 1.02))  # Move offset label slightly

                    ax4.ticklabel_format(axis='both', style='sci', scilimits=(0, 0))  # Force scientific notation

                    ax4.tick_params(axis='both', which='major', labelsize=20)
                    ax4.tick_params(axis='both', which='minor', labelsize=20)

                    ax4.set_ylabel('$t_2$ (s)',fontsize=18)
                    ax4.set_xlabel('$t_1$ (s)',fontsize=18)

                    
                k+=1

        axt, axr = asymmetric_ticks(ax,major_len=major_tick_length, minor_len=minor_tick_length, major_w=major_tick_width, minor_w=minor_tick_width, labelsize=ax_label_font)


fig2.subplots_adjust(wspace=0.25, hspace=-0.67, right=0.90)
cbar = fig2.colorbar(
    im,
    ax=axs,
    pad=0.02,
    fraction=0.035,
    shrink=0.4485,
    aspect=30,
)
cbar.set_label(r'$C(\boldsymbol{Q},t_1,t_2)$', fontsize=22)
cbar.ax.tick_params(
            which="major", length=major_tick_length, width=major_tick_width, labelsize=20
        )
cbar.ax.tick_params(
            which="minor", length=minor_tick_length, width=minor_tick_width, labelsize=20
        )

fig1.savefig(figure_path / "isf.pdf",format="pdf",dpi=300,bbox_inches="tight",metadata={'Creator': ''})

fig2.savefig(figure_path / "twotime.pdf",format="pdf",dpi=300,bbox_inches="tight",metadata={'Creator': ''})

#fig3.savefig(figure_path / "isf_219.pdf",format="pdf",dpi=300,bbox_inches="tight",metadata={'Creator': ''})

#fig4.savefig(figure_path / "twotime_219.pdf",format="pdf",dpi=300,bbox_inches="tight",metadata={'Creator': ''})

print(POPT)
print(PCOV)



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

$\tau$ max scan 219: 44.0
$\tau$ max scan 334: 176.0
$\tau$ max scan 552: 176.0
$\tau$ max scan 579: 224.0
$\tau$ max scan 709: 128.0
$\tau$ max scan 779: 864.0
$\tau$ max scan 796: 768.0
$\tau$ max scan 868: 384.0
[[7.67695353e-01 4.30960946e+01 2.03679757e+00]
 [7.55207868e-01 2.17012268e+02 2.00843892e+00]
 [6.40198575e-01 2.52315589e+02 2.03908401e+00]
 [6.32179698e-01 1.88993263e+02 1.73447676e+00]
 [7.89376554e-01 1.44935268e+02 1.50103126e+00]
 [6.86670811e-01 8.93279601e+02 1.14474271e+00]
 [7.90481241e-01 1.36071959e+03 1.12184850e+00]
 [9.72805782e-01 1.01879871e+04 9.83203735e-01]]
[[8.16110999e-04 6.46873521e-02 7.21660510e-03]
 [2.59385397e-04 2.69259834e-01 4.35788609e-03]
 [2.98988126e-04 5.89003512e-01 7.44811693e-03]
 [1.82603945e-03 1.32529696e+00 2.19953909e-02]
 [5.56109281e-03 2.59389962e+00 4.32866935e-02]
 [1.02254263e-03 4.64028230e+00 7.28598439e-03]
 [3.37672357e-03 3.61723743e+01 2.84352443e-02]
 [7.31056114e-04 1.29327011e+03 3.70906272e-02]]


In [11]:
import matplotlib.pyplot as plt
import numpy as np
%matplotlib notebook

# Three-block errors, same order as T / scans
tau_err = np.array([2.910860, 13.159129, 27.214554, 13.722076, 10.487302, 35.153090, 212.797812, 7108.419518])
beta_err = np.array([0.097725, 0.056227, 0.099997, 0.021122, 0.085878, 0.326819, 0.100013, 0.086681])

fig, ax = plt.subplots(figsize=(7.75,5.75))

tau_color =  '#6a35f0' #'#d54848'        #'#cc5e5e' #'#ce4b5b'
beta_color = '#13aee5' #'#373079'      #'#474273' #'#465373'
# First dataset: Red markers and error bars
ax.semilogy(T, POPT[:,1], 's-', color=tau_color, markersize=marker_size,linewidth=2.5, label=r'$\tau_{F}$',zorder=9)
ax.errorbar(T, POPT[:,1], yerr=tau_err, fmt='none', capsize=3, color=tau_color, zorder=8)

ax.set_ylabel(r'$\tau_r$ (s)', color=tau_color, fontsize=ax_label_font)
ax.set_xlabel('$T$ $(K)$', fontsize=ax_label_font)

# Adjust red y-axis properties
ax.tick_params(axis='y', colors=tau_color)


# --- sizes ---
ax.tick_params(which='major', length=major_tick_length, width=major_tick_width, labelsize=ax_label_font,color=tau_color)
ax.tick_params(which='minor', length=minor_tick_length, width=minor_tick_width,labelsize=ax_label_font,color=tau_color)

ax.tick_params(axis='x',which='major',direction='out',bottom=True, length=major_tick_length, width=major_tick_width, labelsize=ax_label_font,color='k')
ax.tick_params(axis='x',which='minor',direction='out',bottom=True, length=minor_tick_length, width=minor_tick_width,labelsize=ax_label_font,color='k')
# --- directions per side ---
# x: top IN, bottom OUT (two calls so we don't overwrite the other side)
#ax.tick_params(axis='x', which='both',direction='in', top=True)
#ax.tick_params(axis='x', which='both',direction='out', ,color='k')

# y: right IN, left OUT
#ax.spines['left'].set_color('red')
#axt, axr = asymmetric_ticks(ax,major_len=major_tick_length, minor_len=minor_tick_length, major_w=major_tick_width, minor_w=minor_tick_width, labelsize=legend_ax_font)

# Second y-axis for the second dataset
ax2 = ax.twinx()

# Second dataset: Blue markers and error bars
ax2.plot(T, POPT[:,2], '>-', color=beta_color, markersize=marker_size,linewidth=2.5,label=r'$\beta_{F}$',zorder=7)

ax2.errorbar(T, POPT[:,2], yerr=beta_err, fmt='none', capsize=3, color=beta_color, zorder=6)



ax2.set_ylabel(r'$\beta$', color=beta_color, fontsize=ax_label_font)

# Adjust blue y-axis properties
ax2.tick_params(axis='y', colors=beta_color)



# Adjust red y-axis properties
ax2.tick_params(axis='y', colors=beta_color)
ax2.minorticks_on()

# --- sizes ---
ax2.tick_params(which='major', length=major_tick_length, width=major_tick_width, labelsize=ax_label_font,color=beta_color)
ax2.tick_params(which='minor', length=minor_tick_length, width=minor_tick_width,labelsize=ax_label_font,color=beta_color)
#ax2.spines['right'].set_color('blue')

# **Shade the entire bottom half of the figure**
#ax2.axhspan(ymin=0.75, ymax=1, color=beta_color, alpha=0.1)  # Extends fully across x-domain

# **Add a dashed horizontal line at beta = 1**
ax2.axhline(y=1, linestyle='--', color=beta_color, linewidth=2.5,zorder=3)
ax2.axhline(y=2, linestyle='--', color=beta_color, linewidth=2.5,zorder=4)


xmin, xmax = ax.get_xlim()
#ax2.fill_between([287.5, 417.5], 0.75, 1, color=beta_color, alpha=0.3,zorder=100)
#ax2.fill_between([287.5, 417.5], 1, 2, color=beta_color, alpha=0.3,zorder=100)
#ax2.fill_between([287.5, 417.5], 2, 2.25, color=beta_color, alpha=0.3,zorder=100)

ax2.axhspan(0.75, 1, color='#9dbbe0', alpha=0.25,zorder=0)
ax2.axhspan(1, 2, color='#bd91bf', alpha=0.25,zorder=1)
ax2.axhspan(2, 2.25, color="#e87290", alpha=0.25,zorder=2)


ax2.set_ylim(0.75, 2.25)


plt.show()

fig.savefig(figure_path / "tau_beta.pdf",format="pdf",dpi=300,bbox_inches="tight",metadata={'Creator': ''})

<IPython.core.display.Javascript object>

In [ ]:
file = '/Users/your_username/data/PETRA/PETRA_P10_November2023/Beamtime_11017686/XPCS_Scans/Particle_2/298K/NPS_01_00219/e4m/processed/particle2_temp298K_scan219_crop5x5_NewMaskTrue_stabilityFalse.h5'

with h5py.File(file, 'r') as DATA:
    det = DATA['data/det'][:]
    Qx = DATA['pre_processing/Qx'][:]
    Qy = DATA['pre_processing/Qy'][:]

Q_mag = np.sqrt(Qx**2 + Qy**2)

In [13]:
from matplotlib.colors import LogNorm

roi = [430, 820, 180, 570]

def dq_per_pixel_horizontal(Q_mag, roi):
    """Median |Q| step per pixel along columns inside roi [row0, row1, col0, col1]."""
    Q_crop = Q_mag[roi[0]:roi[1], roi[2]:roi[3]]
    dQ = np.abs(np.diff(Q_crop, axis=1))
    n = dQ.shape[0]
    return np.median(dQ[n // 3 : 2 * n // 3, :])

def add_q_scalebar(ax, bar_px, q_label=r"$10^{-2}\,\mathrm{\AA}^{-1}$", margin=(45, 25), color="white", lw=6, fontsize=30):
    """Draw a horizontal Q scale bar in the lower-right of an imshow (data coords)."""
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    x_left, x_right = sorted(xlim)
    y_bottom, y_top = sorted(ylim)  # imshow default origin='upper': larger y = bottom
    h = y_top - y_bottom
    x0 = x_right - bar_px - margin[0]
    y0 = 73.0 #y_top - margin[1]
    x1 = x0 + bar_px
    ax.plot([x0, x1], [y0, y0], color=color, lw=lw, solid_capstyle="butt", clip_on=True)
    ax.text(
        (x0 + x1) / 2-0.02 * h, 13.5,q_label,#y0-0.9*h,q_label,# - 0.14 * h, q_label,
        color=color, ha="center", va="top", fontsize=fontsize,bbox=dict(facecolor='black', alpha=0.75, edgecolor='none', boxstyle='round,pad=0.15'),
        clip_on=True,
    )
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

fig, ax = plt.subplots()


q_bar = 1e-2  # Å^-1
bar_px = q_bar / dq_per_pixel_horizontal(Q_mag, roi)

indices = np.array([0, 1, 2, (det.shape[0] - 1) // 2, det.shape[0] - 1]).T
frame_bbox = None
last_norm = None
for i in indices:
    crop = det[i, roi[0]:roi[1], roi[2]:roi[3]]
    #last_norm = LogNorm(vmin=1e-5, vmax=10)
    ax.imshow(crop , cmap='nipy_spectral', clim=(0,6))
    ax.set_axis_off()
    ax.autoscale(False)

    if frame_bbox is None:
        fig.canvas.draw()
        frame_bbox = ax.get_window_extent().transformed(fig.dpi_scale_trans.inverted())

    if i == indices[-1]:
        add_q_scalebar(ax, bar_px)

    fig.savefig(figure_path / f"det_frame_{i + 1}.pdf", format="pdf", bbox_inches=frame_bbox)
plt.show()

fig_cb, ax_cb = plt.subplots(figsize=(0.50, 4.5))
fig_cb.colorbar(
    plt.cm.ScalarMappable(norm=last_norm, cmap="nipy_spectral"),
    cax=ax_cb,
)
fig_cb.savefig(figure_path / "det_frame_colorbar.pdf", format="pdf", bbox_inches="tight")
plt.show()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>